In [1]:
from io import StringIO
import time
import os
import requests
import pandas as pd
import numpy as np

In [2]:
# URL of the page with the list of IPOS
url = "https://www.iposcoop.com/ipos-recently-filed/"

# Setting the header for the HTTP request
headers = {'User-Agent': 'Mozilla/5.0'}

In [3]:
# Making the HTTP request to get the page content
response = requests.get(url, headers=headers)

# Reading the tables from the HTML page using pandas
tables = pd.read_html(StringIO(response.text))

In [4]:
# Displaying the number of tables found
print(len(tables))   

# Displaying the first rows of the first table
tables[0].head()

1


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-28,Encore Medical Inc.,EMI,The Oak Ridge Financial Services Group/Network...,3.0,$5.00,$5.00,$15.00,Wednesday,S/O
1,2026-09-25,Elevation Acquisition Group,ELEVU,Maxim Group,10.0,$10.00,$10.00,$100.00,TBA,S/O
2,2026-09-25,Holtec Nuclear (Withdrawn),HNUC,J.P.Morgan/Guggenheim Securities/Goldman Sachs...,50.0,$15.00,$18.00,$825.00,Withdrawn,S/O
3,2026-09-25,RegenLab USA,RGNA,StoneX,0.0,NaN,NaN,$35.00,TBA,S/O
4,2026-09-25,TCGX Acquisition Corp. II (Stock-Only),TCXB,Guggenheim Securities,10.0,$10.00,$10.00,$100.00,TBA,S/O


In [5]:
tables

[      File Date                                 Company Symbol  \
 0    2026-09-28                     Encore Medical Inc.    EMI   
 1    2026-09-25             Elevation Acquisition Group  ELEVU   
 2    2026-09-25              Holtec Nuclear (Withdrawn)   HNUC   
 3    2026-09-25                            RegenLab USA   RGNA   
 4    2026-09-25  TCGX Acquisition Corp. II (Stock-Only)   TCXB   
 ..          ...                                     ...    ...   
 495  2026-01-06                Altech Digital Co., Ltd.    ALD   
 496  2026-01-06                 Soren Acquisition Corp.  SORNU   
 497  2026-01-05        Art Technology Acquisition Corp.  ARTCU   
 498  2026-01-05         Black Spade Acquisition III Co.  BIIIU   
 499  2026-01-05                              Buda Juice   BUDA   
 
                                               Managers  Shares (millions)  \
 0    The Oak Ridge Financial Services Group/Network...                3.0   
 1                                    

In [6]:
df = tables[0]

In [7]:
df.to_csv(r'../data/ipo.csv', index=False)

---

### Question 1

In [14]:
df = pd.read_csv(r'../data/ipo.csv')

In [15]:
df.head(3)

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-28,Encore Medical Inc.,EMI,The Oak Ridge Financial Services Group/Network...,3.0,$5.00,$5.00,$15.00,Wednesday,S/O
1,2026-09-25,Elevation Acquisition Group,ELEVU,Maxim Group,10.0,$10.00,$10.00,$100.00,TBA,S/O
2,2026-09-25,Holtec Nuclear (Withdrawn),HNUC,J.P.Morgan/Guggenheim Securities/Goldman Sachs...,50.0,$15.00,$18.00,$825.00,Withdrawn,S/O


In [16]:
file_date_parsed = pd.to_datetime(df['File Date'])
df_witdrawn = df[(df['Expected To Trade'] == 'Withdrawn') & (file_date_parsed < '2026-09-11')]

In [17]:
df_witdrawn.reset_index(inplace=True)

In [18]:
df_witdrawn.head(3)

,index,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,46,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,$100.00,Withdrawn,S/O
1,49,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.0,$4.00,$5.00,$9.00,Withdrawn,S/O
2,56,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.0,NaN,NaN,$100.00,Withdrawn,S/O


In [19]:
conditions = [
    df_witdrawn['Company'].str.contains('Technologies', case=False, na=False),
    df_witdrawn['Company'].str.contains(r'Acquisition Corp|Acquisition Corporation|\bCorp\b|Corp\.', case=False, na=False),
    df_witdrawn['Company'].str.contains('Inc|Incorporated', case=False, na=False),
    df_witdrawn['Company'].str.contains('Group', case=False, na=False),
    df_witdrawn['Company'].str.contains('Ltd|Limited', case=False, na=False),
    df_witdrawn['Company'].str.contains('Holdings|Holding', case=False, na=False)
]

In [20]:
choices = [
    'Technologies',
    'Acquisition Corp',
    'Inc.',
    'Group',
    'Limited',
    'Holdings'
]

In [21]:
df_witdrawn['Company Type'] = np.select(conditions, choices, default='Other')

In [22]:
df_witdrawn.head(3)

,index,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type
0,46,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,$100.00,Withdrawn,S/O,Technologies
1,49,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.0,$4.00,$5.00,$9.00,Withdrawn,S/O,Holdings
2,56,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.0,NaN,NaN,$100.00,Withdrawn,S/O,Other


In [23]:
df_witdrawn.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 31 non-null     int64  
 1   File Date             31 non-null     str    
 2   Company               31 non-null     str    
 3   Symbol                31 non-null     str    
 4   Managers              31 non-null     str    
 5   Shares (millions)     31 non-null     float64
 6   Price Low             25 non-null     str    
 7   Price High            25 non-null     str    
 8   Est $ Vol (millions)  31 non-null     str    
 9   Expected To Trade     31 non-null     str    
 10  SCOOP Rating          31 non-null     str    
 11  Company Type          31 non-null     str    
dtypes: float64(1), int64(1), str(10)
memory usage: 3.0 KB


In [24]:
for col in ['Price High', 'Price Low', 'Est $ Vol (millions)', 'Shares (millions)']:
    df_witdrawn[col] = (
        df_witdrawn[col]
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    df_witdrawn[col] = pd.to_numeric(df_witdrawn[col], errors='coerce')

In [26]:
df_witdrawn['Avg Price'] = (df_witdrawn['Price High'] + df_witdrawn['Price Low']) / 2

In [52]:
df_witdrawn.head(3)

,index,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type,Avg Price,Shares_offered_value
0,46,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,100.0,Withdrawn,S/O,Technologies,NaN,100.0
1,49,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.0,4.0,5.0,9.0,Withdrawn,S/O,Holdings,4.5,9.0
2,56,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.0,NaN,NaN,100.0,Withdrawn,S/O,Other,NaN,100.0


In [28]:
calculated_value = df_witdrawn['Shares (millions)'] * df_witdrawn['Avg Price']

In [29]:
df_witdrawn['Shares_offered_value'] = np.where(
    calculated_value.notna(),
    calculated_value,
    df_witdrawn['Est $ Vol (millions)']
)

In [30]:
df_witdrawn.head(3)

,index,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type,Avg Price,Shares_offered_value
0,46,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,100.0,Withdrawn,S/O,Technologies,NaN,100.0
1,49,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.0,4.0,5.0,9.0,Withdrawn,S/O,Holdings,4.5,9.0
2,56,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.0,NaN,NaN,100.0,Withdrawn,S/O,Other,NaN,100.0


In [31]:
df_grouped = (
    df_witdrawn
    .groupby('Company Type')['Shares_offered_value']
    .sum()
    .reset_index()
)

In [32]:
df_grouped = df_grouped.sort_values(by='Shares_offered_value', ascending=False)

In [33]:
df_grouped

,Company Type,Shares_offered_value
0,Acquisition Corp,499.9850
3,Inc.,351.0000
2,Holdings,311.6575
5,Other,290.4450
4,Limited,197.2500
6,Technologies,184.9000
1,Group,32.5000


---

### Question 2

In [34]:
# Core imports for scraping, timing, and data analysis
from io import StringIO
import time
import os
import requests
import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import pearsonr

In [35]:
url_q2 = 'https://www.iposcoop.com/2025-pricings/'

headers = {'User-Agent': 'Mozilla/5.0'}

# Making the HTTP request to get the page content
response = requests.get(url_q2, headers=headers)

# Reading ALL <table> tables from the HTML page.
# pd.read_html returns a LIST of DataFrames (one entry per table found).
tables = pd.read_html(StringIO(response.text))

# How many tables came back
print(len(tables))

# Peek at each table to figure out which one you want
for i, t in enumerate(tables):
    print(i, t.shape, list(t.columns)[:5])

# Select the 1st table from the list as the working DataFrame
df_ipo = tables[0]

1
0 (231, 10) ['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)']


In [36]:
df_ipo.head(3)

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,Vine Hill Capital Investment Corp. II,VHCPU,Blank Check,12/18/2025,20.0,$10.00,$0.00,$0.00,0.00%,S/O
1,Andersen Group,ANDG,Consumer Services,12/17/2025,11.0,$16.00,$23.50,$23.79,48.69%,S/O
2,Medline Inc.,MDLN,Health Care,12/17/2025,216.0,$29.00,$41.00,$43.83,51.14%,S/O


In [37]:
df_ipo.info()

<class 'pandas.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Company            231 non-null    str    
 1   Symbol             231 non-null    str    
 2   Industry           231 non-null    str    
 3   Offer Date         231 non-null    str    
 4   Shares (millions)  231 non-null    float64
 5   Offer Price        231 non-null    str    
 6   1st Day Close      231 non-null    str    
 7   Current Price      231 non-null    str    
 8   Return             231 non-null    str    
 9   SCOOP Rating       231 non-null    str    
dtypes: float64(1), str(9)
memory usage: 18.2 KB


In [38]:
df_ipo['Offer Date'] = pd.to_datetime(df_ipo['Offer Date'], format='%m/%d/%Y')
ipo = df_ipo[df_ipo['Offer Date'] < '2025-09-01']
ipo = ipo.reset_index(drop=True)

In [39]:
ipo['Return'] = ipo['Return'].str.rstrip('%').astype(float)
ipo = ipo[ipo['Return'] != 0]

In [40]:
ipo.head(3)

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,"Picard Medical, Inc.",PMI,Health Care,2025-08-29,4.3,$4.00,$4.59,$1.68,-58.00,S/O
1,GrowHub Limited (The),TGHL,Technology,2025-08-28,3.7,$4.00,$3.38,$0.40,-90.00,S/O
2,TryHard Holdings Limited,THH,Consumer Services,2025-08-28,1.5,$4.00,$6.02,$29.47,636.75,S/O


In [41]:
tickers_list = ipo['Symbol'].tolist()

result = {}
failed_tickers = []

for ticker in tickers_list:
    data = yf.download(ticker, start='2000-01-01', end='2026-09-12', multi_level_index=False)
    if data.empty:
        failed_tickers.append(ticker)
    else:
        result[ticker] = data

stocks_df = pd.concat(result, names=['Ticker', 'Date']).reset_index()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [42]:
stocks_df['Growth_252d'] = stocks_df['Close'] / stocks_df.groupby('Ticker')['Close'].shift(252)

In [43]:
stocks_df['Volatility'] = stocks_df.groupby('Ticker')['Close'].rolling(30).std().reset_index(level=0, drop=True) * np.sqrt(252)


In [44]:
stocks_df['Sharpe'] = (stocks_df['Growth_252d'] - 0.05) / stocks_df['Volatility']

In [ ]:
final_day = stocks_df[stocks_df['Date'] == '2026-09-11']
print('Number of tickers with data on 2026-09-11:', len(final_day))

display(final_day[['Growth_252d', 'Volatility', 'Sharpe']].describe())

median_sharpe = final_day['Sharpe'].median()
print(f'Median Sharpe on 2026-09-11: {median_sharpe:.4f}')

In [ ]:
mean_growth = final_day['Growth_252d'].mean()
median_growth = final_day['Growth_252d'].median()
print(f'Mean: {mean_growth:.4f}  |  Median: {median_growth:.4f}  |  Mean/median ratio: {mean_growth/median_growth:.2f}x')

final_day[['Ticker', 'Growth_252d']].sort_values('Growth_252d', ascending=False).head(10)

---

### Question 3

In [46]:
for m in range(1, 13):
    days = m * 21
    stocks_df[f'future_growth_{m}_m'] = stocks_df.groupby('Ticker')['Close'].shift(-days) / stocks_df['Close']

In [47]:
stocks_df.head(3)

,Ticker,Date,Close,High,Low,Open,Volume,Growth_252d,Volatility,Sharpe,...,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
0,PMI,2025-08-29,229.5,356.0,209.5,210.5,126794,NaN,NaN,NaN,...,0.464052,0.381264,0.381264,0.209150,0.228758,0.095643,0.058824,0.036819,0.022353,0.016819
1,PMI,2025-09-02,257.0,267.5,232.5,242.5,98358,NaN,NaN,NaN,...,0.412451,0.340467,0.402724,0.221790,0.185019,0.068482,0.059922,0.034047,0.016732,0.017821
2,PMI,2025-09-03,264.5,270.0,250.5,257.0,35960,NaN,NaN,NaN,...,0.374291,0.344045,0.362949,0.202268,0.181474,0.039698,0.060491,0.031758,0.013724,0.016975


In [48]:
min_date_df = stocks_df.loc[
    stocks_df.groupby('Ticker')['Date'].idxmin(),
    ['Ticker', 'Date', 'Close']
].rename(columns={'Date': 'min_date', 'Close': 'min_date_close'})

In [49]:
min_date_df

,Ticker,min_date,min_date_close
58505,AAPG,2025-01-24,17.379999
52890,AARD,2025-02-13,14.310000
42691,ADVB,2025-03-06,73.000000
30494,AII,2025-05-08,16.008226
25970,AIRO,2025-06-13,24.000000
...,...,...,...
13841,WYFI,2025-08-07,16.219999
54078,XHLD,2025-02-13,102.000000
15518,YMAT,2025-07-30,20.400000
1578,YMT,2025-08-19,672.000000


In [50]:
entry_growth_df = stocks_df.merge(
    min_date_df,
    left_on=['Ticker', 'Date'],
    right_on=['Ticker', 'min_date'],
    how='inner'
)

In [ ]:
future_growth_cols = [f'future_growth_{m}_m' for m in range(1, 13)]
stats = entry_growth_df[future_growth_cols].describe()
print(stats)

median_row = stats.loc['50%']
best_month = median_row.idxmax()
print(f'Best month: {best_month} — median: {median_row.max():.4f}')

---

### Question 4

In [56]:
import gdown
import pandas as pd

In [60]:
file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="fastparquet")

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=bf9e3893-c1dd-4d19-a6cb-9822ffbe9497
To: c:\Users\leona\github\stock-markets-analytics-zoomcamp-2026\02-dataframe-analysis\data.parquet
100%|██████████| 130M/130M [00:02<00:00, 43.9MB/s] 


In [61]:
print(df.shape)
print(df.columns.tolist())

(229932, 203)
['Open', 'High', 'Low', 'Close_x', 'Volume', 'Dividends', 'Stock Splits', 'Ticker', 'Year', 'Month', 'Weekday', 'Date', 'growth_1d', 'growth_3d', 'growth_7d', 'growth_30d', 'growth_90d', 'growth_365d', 'growth_future_30d', 'SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative', 'volatility', 'is_positive_growth_30d_future', 'ticker_type', 'index_x', 'adx', 'adxr', 'apo', 'aroon_1', 'aroon_2', 'aroonosc', 'bop', 'cci', 'cmo', 'dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext', 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix', 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo', 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk', 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr', 'index_y', 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase', 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine', 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice', 'index', 

In [62]:
df['Date'] = pd.to_datetime(df['Date'])  # avoids the bug of comparing a date string lexicographically with another string

In [63]:
selected_df = df[
    (df['Date'] >= '2000-01-01') &
    (df['Date'] <= '2025-06-01') &
    (df['rsi'] < 30)
]

In [ ]:
print('Number of signals (trades):', len(selected_df))

net_income = 1000 * (selected_df['growth_future_30d'] - 1).sum()
print(f'Net income: ${net_income:,.2f}  (~${net_income/1000:,.1f}k)')